##### We are getting this warning because we have not specify the model_id that is ok it will default back to distilbert-base-uncased-finetuned-sst-2-english
* Goto https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english

In [ ]:
from transformers import pipeline

help(pipeline)

Help on function pipeline in module transformers.pipelines:

pipeline(task: str = None, model: Union[str, ForwardRef('PreTrainedModel'), ForwardRef('TFPreTrainedModel'), NoneType] = None, config: Union[str, transformers.configuration_utils.PretrainedConfig, NoneType] = None, tokenizer: Union[str, transformers.tokenization_utils.PreTrainedTokenizer, ForwardRef('PreTrainedTokenizerFast'), NoneType] = None, feature_extractor: Union[str, ForwardRef('SequenceFeatureExtractor'), NoneType] = None, image_processor: Union[str, transformers.image_processing_utils.BaseImageProcessor, NoneType] = None, framework: Optional[str] = None, revision: Optional[str] = None, use_fast: bool = True, token: Union[str, bool, NoneType] = None, device: Union[int, str, ForwardRef('torch.device'), NoneType] = None, device_map=None, torch_dtype=None, trust_remote_code: Optional[bool] = None, model_kwargs: Dict[str, Any] = None, pipeline_class: Optional[Any] = None, **kwargs) -> transformers.pipelines.base.Pipeline


In [ ]:
sa_pipeline = pipeline("sentiment-analysis")

sa_pipeline

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
sa_pipeline = pipeline("text-classification")

sa_pipeline

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


In [ ]:
sa_pipeline("The sun is shining, and the birds are singing – what a beautiful day!")

[{'label': 'POSITIVE', 'score': 0.999860405921936}]

In [ ]:
sa_pipeline("I had a flat tire on my way to work today, and it completely ruined my morning.")

[{'label': 'NEGATIVE', 'score': 0.9997496008872986}]

##### Observe this is a binary classifier ie only positive and negative

In [ ]:
sa_pipeline("Today was a typical Monday – nothing extraordinary happened.")

[{'label': 'NEGATIVE', 'score': 0.9981739521026611}]

In [ ]:
sa_pipeline("I finished a book I'd been reading, and it left me with mixed emotions.")

[{'label': 'NEGATIVE', 'score': 0.9965684413909912}]

In [ ]:
sa_pipeline([
    "Dealing with unexpected challenges can be quite frustrating, especially when everything seems to go wrong at once.",
    "I received a surprise gift from a friend today, and it made my whole week!",
    "Meeting new people and making meaningful connections always brings a smile to my face."
])

[{'label': 'NEGATIVE', 'score': 0.9973711967468262},
 {'label': 'POSITIVE', 'score': 0.9998452663421631},
 {'label': 'POSITIVE', 'score': 0.9998735189437866}]

In [ ]:
print(sa_pipeline.model.config.id2label)

{0: 'NEGATIVE', 1: 'POSITIVE'}


Using the tokenizer and model separately

In [ ]:
import tensorflow as tf

from transformers import DistilBertTokenizer, TFAutoModelForSequenceClassification

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

tokenizer

DistilBertTokenizer(name_or_path='distilbert-base-uncased-finetuned-sst-2-english', vocab_size=30522, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

model

All PyTorch model weights were used when initializing TFDistilBertForSequenceClassification.

All the weights of TFDistilBertForSequenceClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForSequenceClassification for predictions without further training.


In [ ]:
inputs = tokenizer("I'd definitely like to do that again", return_tensors="tf")

inputs

{'input_ids': <tf.Tensor: shape=(1, 11), dtype=int32, numpy=
array([[ 101, 1045, 1005, 1040, 5791, 2066, 2000, 2079, 2008, 2153,  102]],
      dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(1, 11), dtype=int32, numpy=array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=int32)>}

In [ ]:
logits = model(**inputs).logits

logits

<tf.Tensor: shape=(1, 2), dtype=float32, numpy=array([[-4.092141 ,  4.3566947]], dtype=float32)>

In [ ]:
predicted_class_id = int(tf.math.argmax(logits, axis=-1)[0])

model.config.id2label[predicted_class_id]

'POSITIVE'

In [ ]:
inputs = tokenizer("I would not recommend that restauratn", return_tensors="tf")

inputs

{'input_ids': <tf.Tensor: shape=(1, 11), dtype=int32, numpy=
array([[  101,  1045,  2052,  2025, 16755,  2008,  2717, 21159,  4017,
         2078,   102]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(1, 11), dtype=int32, numpy=array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=int32)>}

In [ ]:
logits = model(**inputs).logits

logits

<tf.Tensor: shape=(1, 2), dtype=float32, numpy=array([[ 2.7573776, -2.3347328]], dtype=float32)>

In [ ]:
predicted_class_id = int(tf.math.argmax(logits, axis=-1)[0])

model.config.id2label[predicted_class_id]

'NEGATIVE'

### TODO Recording:

* Goto "https://huggingface.co/"
* Click Models from the navbar
* Click "Text Classification" under NLP
* Click on "SamLowe/roberta-base-go_emotions" and check the page (This models is trained to understand emotions)

In [ ]:
sa_pipeline = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions")

sa_pipeline

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
print(sa_pipeline.model.config.id2label)

{0: 'admiration', 1: 'amusement', 2: 'anger', 3: 'annoyance', 4: 'approval', 5: 'caring', 6: 'confusion', 7: 'curiosity', 8: 'desire', 9: 'disappointment', 10: 'disapproval', 11: 'disgust', 12: 'embarrassment', 13: 'excitement', 14: 'fear', 15: 'gratitude', 16: 'grief', 17: 'joy', 18: 'love', 19: 'nervousness', 20: 'optimism', 21: 'pride', 22: 'realization', 23: 'relief', 24: 'remorse', 25: 'sadness', 26: 'surprise', 27: 'neutral'}


In [ ]:
sa_pipeline("The team's hard work and dedication paid off with a successful project completion.")

[{'label': 'admiration', 'score': 0.7227756977081299}]

In [ ]:
sa_pipeline("I'm disappointed with the service I received at the restaurant yesterday.")

[{'label': 'disappointment', 'score': 0.8511288166046143}]

In [ ]:
sa_pipeline("The meeting is scheduled for 2:00 PM.")

[{'label': 'neutral', 'score': 0.9490849375724792}]

In [ ]:
sa_pipeline([
    "The car broke down again, and it caused a lot of inconvenience.",
    "The weather is beautiful, and I'm enjoying a relaxing day outdoors."
])

[{'label': 'disappointment', 'score': 0.4362070858478546},
 {'label': 'joy', 'score': 0.8627153038978577}]

This models is trained specific to stock market: https://huggingface.co/ProsusAI/finbert

In [ ]:
sa_pipeline = pipeline("text-classification", model="ProsusAI/finbert")

sa_pipeline

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
print(sa_pipeline.model.config.id2label)

{0: 'positive', 1: 'negative', 2: 'neutral'}


In [ ]:
sa_pipeline("Stock prices plummeted amid uncertainties surrounding global trade tensions.")

[{'label': 'negative', 'score': 0.9626886248588562}]

In [ ]:
sa_pipeline([
    "Investors are optimistic about the future, contributing to a bullish trend in the stock market with a gain of 150 points.",
    "Investors faced losses due to a bearish market triggered by disappointing corporate results, resulting in a decline of 4%."
])

[{'label': 'positive', 'score': 0.9396426677703857},
 {'label': 'negative', 'score': 0.9742977619171143}]

##### We are getting neutral as it is not related to stocks

In [ ]:
sa_pipeline("I'm happy today!")

[{'label': 'neutral', 'score': 0.6555015444755554}]

In [ ]:
sa_pipeline("The weather is beautiful, and I'm enjoying a relaxing day outdoors.")

[{'label': 'neutral', 'score': 0.8601391911506653}]